# NB8 — Error por clase AAMI y detección de evento

---

## Qué hace este cuaderno

Reejecuta el protocolo *Leave-One-Patient-Out* de la sexta fase y, en cada pliegue, registra
para **cada latido predicho** del paciente excluido: su error, su fidelidad de forma y su
**clase AAMI**. Sobre esa tabla se construyen los dos análisis pendientes.

Además guarda **las formas de onda predichas**, de modo que la misma corrida sirva para un
tercer análisis: aplicar un clasificador de arritmias externo sobre la señal predicha.
Sin esto haría falta repetir toda la ejecución.

## Por qué hay que reejecutar el protocolo

En el directorio de modelos solo se conserva `CNN_GRU_ATTN_final.keras`, entrenado con los
**123 pacientes**. Emplearlo para evaluar a cualquiera de ellos sería **fuga de información**,
exactamente el sesgo que la tercera fase del proyecto documentó y criticó. Los modelos de cada
pliegue no se guardaron.

## Correcciones respecto de la versión anterior de este cuaderno

Se contrastó línea a línea contra `06_Cross_patient_MultiStep.ipynb`, que es la referencia.
Nueve divergencias corregidas:

| # | Qué estaba mal | Consecuencia |
| :---- | :---- | :---- |
| 1 | INCART no se remuestreaba de 257 Hz a 360 Hz | La ventana de 256 muestras cubría 1.00 s en vez de 0.71 s. Afectaba a los **75 pacientes de INCART**, el 61 % de la cohorte |
| 2 | El intervalo RR se normalizaba con constantes fijas de los 123 pacientes | **Fuga de información** dentro del protocolo LOPO. La sexta fase recalcula esas estadísticas en cada pliegue, solo con los 122 de entrenamiento |
| 3 | Recorte del latido a ±8 | La sexta fase recorta a **±5** |
| 4 | El intervalo RR no se recortaba | La sexta fase lo recorta a **[0.3, 2.0] s** |
| 5 | `n = len(beats) − LOOKBACK − HORIZON` | Faltaba el `+ 1`: se perdía una ventana por paciente |
| 6 | Sin guarda de mínimo de picos R | La sexta fase exige **al menos 5** |
| 7 | `validation_split=0.1` con parada temprana | La sexta fase entrena **30 épocas fijas sin validación**. Además el 10 % final del conjunto concatenado son pacientes completos, no una muestra representativa |
| 8 | Submuestreo con los **primeros** 400 pares | La sexta fase toma **400 al azar**; los primeros sesgan hacia el inicio del registro |
| 9 | Sin aumentación de datos | La sexta fase aplica `augmentar_multi` al conjunto de entrenamiento tras construirlo |

También se añadieron la instalación explícita de dependencias, comprobaciones de entorno que
fallan pronto y con un mensaje claro, y una celda de verificación de fidelidad que contrasta la
configuración contra los metadatos del modelo entrenado.

## 1. Entorno y dependencias

`wfdb` no viene preinstalado en Colab. TensorFlow sí, pero conviene dejar constancia de la versión.

In [1]:
# Dependencias. En Colab wfdb NO viene preinstalado.
import importlib, subprocess, sys

def asegurar(paquete, importable=None):
    nombre = importable or paquete
    try:
        importlib.import_module(nombre)
        return False
    except ImportError:
        print('Instalando %s ...' % paquete)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paquete])
        return True

asegurar('wfdb')
asegurar('scikit-learn', 'sklearn')
asegurar('scipy')
asegurar('pandas')
asegurar('matplotlib')
print('Dependencias listas.')

Instalando wfdb ...
Dependencias listas.


In [2]:
import os, sys, json, time, gc, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.signal as spsig
from scipy.stats import kruskal, mannwhitneyu
from sklearn.metrics import (r2_score, precision_recall_curve,
                             average_precision_score, confusion_matrix)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import wfdb

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

print('Python     ', sys.version.split()[0])
print('TensorFlow ', tf.__version__)
print('Keras      ', keras.__version__)
print('NumPy      ', np.__version__)
print('wfdb       ', wfdb.__version__)

_gpu = tf.config.list_physical_devices('GPU')
print('GPU        ', _gpu if _gpu else 'NO DISPONIBLE')
if not _gpu:
    print('\nAVISO: sin GPU cada pliegue tarda horas en vez de minutos.')
    print('       En Colab: Entorno de ejecucion > Cambiar tipo de entorno > GPU.')

Python      3.13.15
TensorFlow  2.20.0
Keras       3.13.2
NumPy       2.1.3
wfdb        4.3.1
GPU         [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuración — idéntica a la sexta fase

Todas las constantes se copian de `06_Cross_patient_MultiStep.ipynb`. Ninguna se reinterpreta.

In [3]:
# ── Señal ──────────────────────────────────────────────────────────────
FS = 360
NYQUIST = FS / 2
F_BAJO, F_ALTO, ORDEN_BUT = 0.5, 40.0, 4
FREC_NOTCH, Q_NOTCH = 60.0, 30.0

# ── Segmentación ───────────────────────────────────────────────────────
BEAT_LEN = 256
N_LOOKBACK = 5
HORIZON = 3
PRE_R = BEAT_LEN * 36 // 100          # 92
POST_R = BEAT_LEN - PRE_R             # 164
BEAT_TYPES = set('NLRBVAFaJfEjeSe/')

# ── Rasgos ─────────────────────────────────────────────────────────────
INCLUDE_RR = True
FEAT_DIM = BEAT_LEN + (1 if INCLUDE_RR else 0)

# ── Conjunto de datos ──────────────────────────────────────────────────
MAX_PARES_POR_PACIENTE = 400
AUG_FACTOR = 0.3

# ── Entrenamiento ──────────────────────────────────────────────────────
EPOCHS_BASE = 30
BATCH_BASE = 32
LR_BASE = 1e-3
DROPOUT_BASE = 0.3

# ── Modo de ejecución ──────────────────────────────────────────────────
MODO_PRUEBA = False   # True = 4 pliegues para validar el flujo; False = la cohorte completa

assert PRE_R == 92 and POST_R == 164, 'La ventana no coincide con la sexta fase'
assert FEAT_DIM == 257, 'La dimension de entrada no coincide con el modelo entrenado'
print('Ventana: PRE_R=%d  POST_R=%d  BEAT_LEN=%d  FEAT_DIM=%d' % (PRE_R, POST_R, BEAT_LEN, FEAT_DIM))
print('Entrada (%d, %d) -> salida (%d, %d)' % (N_LOOKBACK, FEAT_DIM, HORIZON, BEAT_LEN))

Ventana: PRE_R=92  POST_R=164  BEAT_LEN=256  FEAT_DIM=257
Entrada (5, 257) -> salida (3, 256)


In [4]:
# Correspondencia simbolo PhysioNet -> clase AAMI (ANSI/AAMI EC57).
# Identica a scripts/p2_analisis_exploratorio.py.
AAMI = {}
for _c in 'NLRej': AAMI[_c] = 'N'      # normal o de conduccion
for _c in 'AaJS':  AAMI[_c] = 'SVEB'   # ectopico supraventricular
for _c in 'VE':    AAMI[_c] = 'VEB'    # ectopico ventricular
for _c in 'F':     AAMI[_c] = 'F'      # fusion
for _c in '/fQ':   AAMI[_c] = 'Q'      # no clasificable o estimulado

# 'B' pertenece a BEAT_TYPES pero no a la norma AAMI: queda como no clasificable.
SIN_CLASE_AAMI = sorted(BEAT_TYPES - set(AAMI))
print('Simbolos sin clase AAMI explicita (se agrupan en Q):', SIN_CLASE_AAMI)

CLASES = ['N', 'SVEB', 'VEB', 'F', 'Q']
ECTOPICAS = ['SVEB', 'VEB', 'F']       # la clase Q se excluye del bloque P4

Simbolos sin clase AAMI explicita (se agrupan en Q): ['B']


In [5]:
def configurar_rutas():
    try:
        import google.colab                      # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        base = '/content/drive/MyDrive/FORECASTING ECG'
        entorno = 'colab'
    except ImportError:
        entorno = 'local'
        base = os.getcwd()
        for _ in range(5):
            if os.path.isdir(os.path.join(base, 'datos')):
                break
            padre = os.path.dirname(base)
            if padre == base:
                break
            base = padre
    return {
        'ENTORNO': entorno,
        'BASE': base,
        'MITBIH': os.path.join(base, 'datos', 'mit-bih'),
        'INCART': os.path.join(base, 'datos', 'incart'),
        'SALIDA': os.path.join(base, 'results', 'NB8'),
        'ONDAS': os.path.join(base, 'results', 'NB8', 'ondas_predichas'),
    }

RUTAS = configurar_rutas()
os.makedirs(RUTAS['SALIDA'], exist_ok=True)
os.makedirs(RUTAS['ONDAS'], exist_ok=True)
print(json.dumps(RUTAS, indent=2, ensure_ascii=False))

# Falla pronto y con mensaje claro si los datos no estan donde se espera.
for _clave in ('MITBIH', 'INCART'):
    if not os.path.isdir(RUTAS[_clave]):
        raise FileNotFoundError(
            'No se encuentra la carpeta de datos %s en %s.\n'
            'Coloca los registros de PhysioNet ahi, o corrige configurar_rutas().'
            % (_clave, RUTAS[_clave]))
print('\nCarpetas de datos verificadas.')

Mounted at /content/drive
{
  "ENTORNO": "colab",
  "BASE": "/content/drive/MyDrive/FORECASTING ECG",
  "MITBIH": "/content/drive/MyDrive/FORECASTING ECG/datos/mit-bih",
  "INCART": "/content/drive/MyDrive/FORECASTING ECG/datos/incart",
  "SALIDA": "/content/drive/MyDrive/FORECASTING ECG/results/NB8",
  "ONDAS": "/content/drive/MyDrive/FORECASTING ECG/results/NB8/ondas_predichas"
}

Carpetas de datos verificadas.


## 3. Carga y preprocesamiento

La cadena es notch → paso banda → eliminación de tendencia → z-score global. **No incluye filtro
de mediana**, por las razones establecidas en la TABLA XXXV del documento.

Los registros de INCART se **remuestrean de 257 Hz a 360 Hz antes de filtrar**, y las marcas de
anotación se reescalan con ellos. Sin ese paso la ventana de 256 muestras abarcaría una duración
distinta y la morfología no sería comparable entre bases.

### Dos salvedades heredadas de la sexta fase, que se conservan a propósito

**Derivación.** Se toma siempre el **canal 0**, igual que la sexta fase. El documento declara la
derivación MLII para MIT-BIH, y eso es cierto en 45 de los 48 registros. En los otros tres no:
en **102** y **104** el canal 0 es V5 y MLII no existe en el registro, de modo que no hay
alternativa; en **114** el canal 0 es V5 pero **MLII sí está disponible en el canal 1**. Los tres
pertenecen a la cohorte evaluada.

No se corrige aquí. Cambiar la derivación del registro 114 apartaría estos datos de la
distribución con la que se entrenó el modelo, y la comparación dejaría de ser válida. Queda
documentado como limitación, que es lo que corresponde declarar en el documento.

**Alineamiento del pico R.** El pico R debe caer en el índice 92 de cada ventana. Medido sobre el
promedio de latidos, así ocurre con desviaciones de −2 a +3 muestras según el paciente, atribuibles
al convenio de anotación y al redondeo del remuestreo. La excepción es el registro **114**, con un
desplazamiento de +12 muestras, consecuencia directa de la salvedad anterior.

In [6]:
def filtro_completo(s):
    """Cadena F_NB6. Opera siempre a FS = 360 Hz: la senal ya llega remuestreada."""
    s = s.astype(np.float32)
    try:
        b, a = spsig.iirnotch(FREC_NOTCH, Q_NOTCH, FS)
        s = spsig.filtfilt(b, a, s)
        sos = spsig.butter(ORDEN_BUT, [F_BAJO / NYQUIST, F_ALTO / NYQUIST],
                           btype='band', output='sos')
        s = spsig.sosfiltfilt(sos, s)
    except Exception as e:
        print('  aviso, filtro degradado:', e)
    s = spsig.detrend(s, type='linear')
    s = (s - np.mean(s)) / (np.std(s) + 1e-8)
    return np.nan_to_num(s).astype(np.float32)


def cargar_registro(pid):
    """Devuelve (senal_filtrada, ann) con INCART ya remuestreado a 360 Hz."""
    es_incart = str(pid).startswith('I')
    ruta = os.path.join(RUTAS['INCART' if es_incart else 'MITBIH'], str(pid))
    if not os.path.exists(ruta + '.dat'):
        return None, None
    try:
        rec = wfdb.rdrecord(ruta)
        s = (rec.p_signal[:, 0] if rec.p_signal.ndim > 1 else rec.p_signal).astype(np.float32)
        ann = wfdb.rdann(ruta, 'atr')

        fs_orig = int(rec.fs)
        if fs_orig != FS:
            s = spsig.resample(s, int(len(s) * FS / fs_orig)).astype(np.float32)
            ann.sample = (ann.sample * FS / fs_orig).astype(np.int64)

        return filtro_completo(s), ann
    except Exception as e:
        print('  ERROR [%s]: %s' % (pid, e))
        return None, None


def revisar_derivaciones(pids):
    """Deja constancia de los registros cuyo canal 0 no es MLII."""
    avisos = []
    for pid in pids:
        if str(pid).startswith('I'):
            continue
        ruta = os.path.join(RUTAS['MITBIH'], str(pid))
        if not os.path.exists(ruta + '.hea'):
            continue
        try:
            cab = wfdb.rdheader(ruta)
            if cab.sig_name and cab.sig_name[0] != 'MLII':
                avisos.append((str(pid), list(cab.sig_name)))
        except Exception:
            pass
    return avisos

## 4. Segmentación con el símbolo de anotación

Única ampliación respecto de la sexta fase: `extraer_latidos_sym` devuelve también el símbolo de
cada latido, necesario para asignarle su clase AAMI. La aritmética de la ventana, la normalización
por latido y el recorte son idénticos.

In [7]:
def extraer_latidos_sym(senal, ann):
    """Devuelve (beats, rr, simbolos) alineados uno a uno.

    Identico a extraer_latidos_v3 de la sexta fase, con el simbolo anadido.
    """
    vacio = (np.empty((0, BEAT_LEN), np.float32), np.empty((0,), np.float32), [])
    if ann is None:
        return vacio

    mascara = [sym in BEAT_TYPES for sym in ann.symbol]
    rpeaks = np.asarray(ann.sample, dtype=np.int64)[mascara]
    simbolos = [s for s, m in zip(ann.symbol, mascara) if m]

    if len(rpeaks) < 5:
        return vacio

    beats, rr_ints, syms = [], [], []
    for i in range(1, len(rpeaks) - 1):
        r = int(rpeaks[i])
        ini, fin = r - PRE_R, r + POST_R
        if ini < 0 or fin > len(senal):
            continue
        beat = senal[ini:fin].astype(np.float32)
        if len(beat) != BEAT_LEN:
            continue

        mu = float(np.mean(beat))
        sd = max(float(np.std(beat)), 1e-6)
        beat = np.clip((beat - mu) / sd, -5, 5)

        rr = float(rpeaks[i] - rpeaks[i - 1]) / FS
        rr = float(np.clip(rr, 0.3, 2.0))

        beats.append(beat)
        rr_ints.append(rr)
        syms.append(simbolos[i])

    if not beats:
        return vacio
    return (np.array(beats, np.float32), np.array(rr_ints, np.float32), syms)


def secuencias_multistep_sym(beats, rr, syms):
    """X=(n,5,257) con el RR SIN normalizar, Y=(n,3,256), S=(n,3) clases AAMI.

    El RR se normaliza despues, con las estadisticas del conjunto de entrenamiento
    de cada pliegue. Normalizarlo aqui con constantes globales seria fuga de informacion.
    """
    n = len(beats) - N_LOOKBACK - HORIZON + 1
    if n <= 0:
        return (np.empty((0, N_LOOKBACK, FEAT_DIM), np.float32),
                np.empty((0, HORIZON, BEAT_LEN), np.float32),
                np.empty((0, HORIZON), dtype='<U4'))

    X = np.zeros((n, N_LOOKBACK, FEAT_DIM), np.float32)
    Y = np.zeros((n, HORIZON, BEAT_LEN), np.float32)
    S = np.empty((n, HORIZON), dtype='<U4')

    for i in range(n):
        ventana = beats[i:i + N_LOOKBACK]
        if INCLUDE_RR:
            ventana = np.concatenate([ventana, rr[i:i + N_LOOKBACK].reshape(-1, 1)], axis=1)
        X[i] = ventana
        Y[i] = beats[i + N_LOOKBACK: i + N_LOOKBACK + HORIZON]
        for h in range(HORIZON):
            S[i, h] = AAMI.get(syms[i + N_LOOKBACK + h], 'Q')
    return X, Y, S


def augmentar_multi(X, Y, rng):
    """Aumentacion identica a la sexta fase: ruido leve y reescalado de amplitud."""
    n_aug = int(len(X) * AUG_FACTOR)
    if n_aug == 0:
        return X, Y
    idx = rng.choice(len(X), n_aug, replace=True)
    Xa = X[idx].copy().astype(np.float32)
    Ya = Y[idx].copy().astype(np.float32)
    Xa[:, :, :BEAT_LEN] += rng.normal(0, 0.003, Xa[:, :, :BEAT_LEN].shape)
    escala = rng.uniform(0.97, 1.03, (len(Xa), 1, 1)).astype(np.float32)
    Xa[:, :, :BEAT_LEN] *= escala
    Ya *= escala
    Xa = np.clip(Xa, -5, 5)
    Ya = np.clip(Ya, -5, 5)
    return (np.concatenate([X, Xa]).astype(np.float32),
            np.concatenate([Y, Ya]).astype(np.float32))

## 5. Arquitectura y función de pérdida — copia literal de la sexta fase

La capa `Lambda` declara explícitamente su forma de salida: sin eso, Keras 3 no puede inferirla y
la construcción del modelo falla.

In [8]:
def ecg_loss(y_true, y_pred):
    mse = tf.reduce_mean(tf.square(y_true - y_pred))
    mae = tf.reduce_mean(tf.abs(y_true - y_pred))
    d_true = y_true[:, :, 1:] - y_true[:, :, :-1]
    d_pred = y_pred[:, :, 1:] - y_pred[:, :, :-1]
    slope = tf.reduce_mean(tf.square(d_true - d_pred))
    return 0.5 * mse + 0.3 * mae + 0.2 * slope


@keras.utils.register_keras_serializable(package='nb8')
class TemporalAttention(layers.Layer):
    def build(self, input_shape):
        n = int(input_shape[-1])
        self.W = self.add_weight(name='att_W', shape=(n, 1), initializer='glorot_uniform')
        self.b = self.add_weight(name='att_b', shape=(1,), initializer='zeros')
        super().build(input_shape)

    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        return tf.reduce_sum(x * a, axis=1)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        return super().get_config()


def build_cnn_gru_attn_multi(input_shape, out_dim=BEAT_LEN, horizon=HORIZON):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(128, 3, padding='same', dilation_rate=2, activation='relu')(x)
    x = layers.Conv1D(128, 3, padding='same', dilation_rate=4, activation='relu')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.GRU(128, return_sequences=True)(x)
    x = layers.GRU(64, return_sequences=True)(x)
    x = TemporalAttention()(x)
    x = layers.Dropout(DROPOUT_BASE)(x)
    h = layers.Dense(128, activation='relu')(x)

    # Conexion residual: el ultimo latido observado mas un incremento aprendido.
    ultimo = layers.Lambda(lambda t: t[:, -1, :out_dim],
                           output_shape=(out_dim,))(inp)
    ultimo = layers.RepeatVector(horizon)(ultimo)
    delta = layers.Dense(horizon * out_dim)(h)
    delta = layers.Reshape((horizon, out_dim))(delta)
    return keras.Model(inp, layers.Add(name='out_ecg')([ultimo, delta]),
                       name='CNN_GRU_ATTN')


# Comprobacion de formas antes de gastar horas de computo.
_m = build_cnn_gru_attn_multi((N_LOOKBACK, FEAT_DIM))
_salida = _m.predict(np.zeros((2, N_LOOKBACK, FEAT_DIM), np.float32), verbose=0)
assert _salida.shape == (2, HORIZON, BEAT_LEN), 'Forma de salida inesperada: %s' % (_salida.shape,)
print('Modelo construido. Parametros: %s' % format(_m.count_params(), ','))
print('Entrada (%d, %d) -> salida %s  correcto' % (N_LOOKBACK, FEAT_DIM, _salida.shape[1:]))
del _m, _salida
keras.backend.clear_session()

Modelo construido. Parametros: 400,321
Entrada (5, 257) -> salida (3, 256)  correcto


## 6. Carga de la cohorte

Se cargan los **123 pacientes**: son el conjunto de entrenamiento de cada pliegue. Se guardan los
latidos, los intervalos RR y los símbolos, no las ventanas ya construidas, porque la normalización
del RR depende del pliegue.

In [ ]:
MITBIH = ['100','101','102','103','104','105','106','107','108','109',
          '111','112','113','114','115','116','117','118','119',
          '121','122','123','124','200','201','202','203','205','207','208',
          '209','210','212','213','214','215','217','219','220','221','222',
          '223','228','230','231','232','233','234']
INCART = ['I%02d' % i for i in range(1, 76)]
TODOS = MITBIH + INCART

assert len(MITBIH) == 48 and len(INCART) == 75 and len(TODOS) == 123

print('Cargando %d pacientes...' % len(TODOS))
DATOS = {}
t0 = time.time()
for k, pid in enumerate(TODOS, 1):
    senal, ann = cargar_registro(pid)
    if senal is None:
        continue
    beats, rr, syms = extraer_latidos_sym(senal, ann)
    if len(beats) >= (N_LOOKBACK + HORIZON):
        DATOS[pid] = (beats, rr, syms)
    if k % 25 == 0:
        print('  %3d/%d  (%.0f s)' % (k, len(TODOS), time.time() - t0))
    del senal, ann

_m = sum(1 for p in DATOS if not p.startswith('I'))
print('\n%d pacientes cargados en %.1f min  (%d MIT-BIH, %d INCART)'
      % (len(DATOS), (time.time() - t0) / 60, _m, len(DATOS) - _m))
print('Latidos totales: %s' % format(sum(len(v[0]) for v in DATOS.values()), ','))

if len(DATOS) < 100:
    print('\nAVISO: se esperaban del orden de 123 pacientes. Revisa las rutas de datos.')

# Salvedad de la derivacion: se deja constancia, no se corrige (ver seccion 3).
_avisos = revisar_derivaciones(DATOS.keys())
if _avisos:
    print('\nRegistros de MIT-BIH cuyo canal 0 no es MLII (se usa el canal 0, como la sexta fase):')
    for _pid, _canales in _avisos:
        _hay = 'MLII disponible en el canal %d' % _canales.index('MLII') if 'MLII' in _canales \
               else 'sin MLII en el registro'
        print('  %-5s canales=%-14s %s' % (_pid, ','.join(_canales), _hay))
    json.dump([{'paciente': p, 'canales': c} for p, c in _avisos],
              open(os.path.join(RUTAS['SALIDA'], 'salvedad_derivaciones.json'), 'w'), indent=1)

Cargando 123 pacientes...
   25/123  (78 s)
   50/123  (156 s)


In [ ]:
# Verificacion de fidelidad contra los metadatos del modelo entrenado.
_meta = os.path.join(RUTAS['BASE'], 'results', 'NB5', 'modelos',
                     'CNN_GRU_ATTN_final_metadata.json')
if os.path.exists(_meta):
    M = json.load(open(_meta, encoding='utf-8'))
    esperado = {'lookback': N_LOOKBACK, 'horizon': HORIZON, 'beat_len': BEAT_LEN,
                'feat_dim': FEAT_DIM, 'fs': FS, 'include_rr': INCLUDE_RR}
    print('%-12s %-12s %-12s %s' % ('clave', 'cuaderno', 'metadatos', 'coincide'))
    todo_ok = True
    for k, v in esperado.items():
        ref = M.get(k)
        igual = (ref == v)
        todo_ok = todo_ok and igual
        print('%-12s %-12s %-12s %s' % (k, v, ref, 'si' if igual else 'NO'))
    print('\nFidelidad de configuracion:', 'CORRECTA' if todo_ok else 'HAY DIVERGENCIAS')
else:
    print('No se encontro el metadata del modelo final; se omite la comprobacion.')
    print('Ruta esperada:', _meta)

## 7. Definición de los pliegues

El conjunto de entrenamiento se mantiene completo: en cada pliegue el modelo se entrena con los
122 pacientes restantes, exactamente igual que en la sexta fase. Reducir el conjunto de
entrenamiento cambiaría el resultado y haría incomparables las cifras.

Lo que se reduce es **el número de pliegues evaluados**: 50 pacientes en lugar de 123, mediante la
cohorte estratificada de `results/NB7/cohorte_reducida_50.json`. El coste baja de 26.5 h a 10.8 h
y el error típico de la estimación del R² medio es de ±0.0196.

In [ ]:
COHORTE_EVAL = ['102','104','107','109','114','115','117','121','122','123','124',
                '203','205','208','212','217','220','221','228','234',
                'I03','I07','I08','I11','I12','I20','I21','I25','I26','I29','I30',
                'I31','I32','I34','I35','I39','I40','I49','I50','I51','I52','I53',
                'I58','I59','I61','I62','I67','I69','I71','I75']
assert len(COHORTE_EVAL) == 50

_faltan = [p for p in COHORTE_EVAL if p not in DATOS]
if _faltan:
    print('AVISO: no se cargaron estos pacientes de la cohorte:', _faltan)

if MODO_PRUEBA:
    _cand = ['104', '203', 'I31', 'I61']       # 2 por base, incluye el peor caso
    FOLDS = [p for p in _cand if p in DATOS]
    print('MODO PRUEBA: %d pliegues -> %s' % (len(FOLDS), FOLDS))
    print('Sirve para validar el flujo completo. Poner MODO_PRUEBA = False para la corrida real.')
else:
    FOLDS = [p for p in COHORTE_EVAL if p in DATOS]
    _m = sum(1 for p in FOLDS if not p.startswith('I'))
    print('Pliegues a evaluar: %d  (%d MIT-BIH, %d INCART)' % (len(FOLDS), _m, len(FOLDS) - _m))
    print('Entrenamiento por pliegue: %d pacientes' % (len(DATOS) - 1))
    print('Tiempo estimado: %.1f h  (a 774 s por pliegue, medido en la sexta fase)'
          % (len(FOLDS) * 774.3 / 3600))

assert FOLDS, 'No hay ningun pliegue evaluable: revisa la carga de datos.'

## 8. Bucle LOPO

Por cada pliegue:

1. Se construye el conjunto de entrenamiento con los 122 pacientes restantes, tomando **400 pares
   al azar** por paciente.
2. Se calculan la media y la desviación del intervalo RR **solo sobre esos 122** y se aplican tanto
   al entrenamiento como al paciente evaluado. Así no hay fuga de información.
3. Se aumenta el conjunto de entrenamiento, igual que en la sexta fase.
4. Se entrena 30 épocas fijas, sin validación ni parada temprana.
5. Se predice sobre **todas** las ventanas del paciente excluido, sin límite.
6. Se guardan el error por latido con su clase AAMI, y las formas de onda predichas.

### Resistencia a la caída de la sesión

Colab desconecta por inactividad, por agotar la cuota de GPU o sin previo aviso. Una corrida de
once horas **se va a cortar**. El guardado está diseñado para que no se pierda nada de lo ya hecho:

- **Un archivo por paciente**, no un CSV compartido al que se va añadiendo. Así una caída a mitad de
  escritura no puede corromper el trabajo de los pliegues anteriores.
- **Escritura atómica**: cada archivo se escribe primero con extensión `.tmp` y solo entonces se
  renombra. Un archivo con nombre definitivo está siempre completo.
- **Orden de compromiso**: primero las formas de onda, después el CSV del paciente. Un pliegue se
  da por hecho solo si existen **los dos** archivos. Si la sesión muere entre uno y otro, el pliegue
  se repite entero en vez de quedar a medias.
- **Reanudación automática**: al reejecutar, se detectan los pliegues completos y se saltan.
- **Registro de progreso** en `progreso.json`, con el tiempo de cada pliegue y la estimación de lo
  que falta.

Si trabajas en Colab, la carpeta de salida **debe estar en Google Drive**. Si apunta a `/content`,
todo se borra al desconectar; la siguiente celda lo comprueba y avisa.

In [ ]:
DIR_ERR = os.path.join(RUTAS['SALIDA'], 'errores_por_paciente')
PROGRESO = os.path.join(RUTAS['SALIDA'], 'progreso.json')
os.makedirs(DIR_ERR, exist_ok=True)


def ruta_errores(pid):
    return os.path.join(DIR_ERR, 'errores_%s.csv' % pid)


def ruta_ondas(pid):
    return os.path.join(RUTAS['ONDAS'], 'ondas_%s.npz' % pid)


def guardar_csv_atomico(df, ruta):
    """Escribe en .tmp y renombra. Un archivo con nombre definitivo esta completo."""
    tmp = ruta + '.tmp'
    df.to_csv(tmp, index=False)
    os.replace(tmp, ruta)


def guardar_npz_atomico(ruta, **arrays):
    tmp = ruta + '.tmp.npz'
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, ruta)


def pliegues_completos():
    """Un pliegue esta hecho solo si existen SUS DOS archivos."""
    hechos = set()
    for f in os.listdir(DIR_ERR):
        if f.startswith('errores_') and f.endswith('.csv'):
            pid = f[len('errores_'):-len('.csv')]
            if os.path.exists(ruta_ondas(pid)):
                hechos.add(pid)
    return hechos


def limpiar_temporales():
    """Restos de una caida a mitad de escritura. Se borran sin riesgo."""
    n = 0
    for carpeta in (DIR_ERR, RUTAS['ONDAS']):
        for f in os.listdir(carpeta):
            if f.endswith('.tmp') or f.endswith('.tmp.npz'):
                os.remove(os.path.join(carpeta, f)); n += 1
    return n


def registrar_progreso(pid, minutos, r2, n_lat):
    hist = {}
    if os.path.exists(PROGRESO):
        try:
            hist = json.load(open(PROGRESO, encoding='utf-8'))
        except Exception:
            hist = {}
    hist[str(pid)] = {'minutos': round(minutos, 2), 'r2': round(float(r2), 4),
                      'latidos': int(n_lat), 'cuando': time.strftime('%Y-%m-%d %H:%M:%S')}
    tmp = PROGRESO + '.tmp'
    json.dump(hist, open(tmp, 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
    os.replace(tmp, PROGRESO)
    return hist


# ── Comprobacion de persistencia: en Colab la salida debe estar en Drive ──
if RUTAS['ENTORNO'] == 'colab' and '/content/drive' not in RUTAS['SALIDA'].replace('\\', '/'):
    print('AVISO GRAVE: la carpeta de salida NO esta en Google Drive.')
    print('             %s' % RUTAS['SALIDA'])
    print('             Todo lo calculado se perdera al desconectar la sesion.')
else:
    print('Persistencia verificada. Salida en:', RUTAS['SALIDA'])

_borrados = limpiar_temporales()
if _borrados:
    print('Se eliminaron %d archivos temporales de una interrupcion anterior.' % _borrados)


def construir_pool(pids, rng, max_pares=MAX_PARES_POR_PACIENTE):
    """Conjunto de entrenamiento del pliegue. Devuelve X, Y y las estadisticas del RR."""
    Xs, Ys, rrs = [], [], []
    for pid in pids:
        beats, rr, syms = DATOS[pid]
        if len(beats) < (N_LOOKBACK + HORIZON):
            continue
        X, Y, _ = secuencias_multistep_sym(beats, rr, syms)
        if len(X) == 0:
            continue
        rrs.append(rr)                      # antes de submuestrear, como la sexta fase
        if max_pares is not None and len(X) > max_pares:
            idx = rng.choice(len(X), max_pares, replace=False)
            X, Y = X[idx], Y[idx]
        Xs.append(X); Ys.append(Y)

    if not Xs:
        raise ValueError('No se pudo construir el conjunto de entrenamiento')

    X_all = np.concatenate(Xs).astype(np.float32)
    Y_all = np.concatenate(Ys).astype(np.float32)
    rr_all = np.concatenate(rrs)
    rr_mu = float(np.mean(rr_all))
    rr_std = float(np.std(rr_all) + 1e-8)
    if INCLUDE_RR:
        X_all[:, :, -1] = (X_all[:, :, -1] - rr_mu) / rr_std
    return X_all, Y_all, rr_mu, rr_std


def construir_test(pid, rr_mu, rr_std):
    """Todas las ventanas del paciente evaluado, con el RR normalizado con las del pool."""
    beats, rr, syms = DATOS[pid]
    X, Y, S = secuencias_multistep_sym(beats, rr, syms)
    if len(X) and INCLUDE_RR:
        X[:, :, -1] = (X[:, :, -1] - rr_mu) / rr_std
    return X, Y, S


hechos = pliegues_completos()
if hechos:
    print('\nReanudando: %d pliegues ya completos -> %s' % (len(hechos), sorted(hechos)))
else:
    print('\nSin trabajo previo: se empieza desde cero.')

In [ ]:
# Se recalcula aqui: asi esta celda se puede reejecutar sola tras una caida.
hechos = pliegues_completos()
pendientes = [p for p in FOLDS if str(p) not in hechos]
print('Pliegues: %d totales, %d hechos, %d pendientes' % (len(FOLDS), len(FOLDS) - len(pendientes), len(pendientes)))
if pendientes:
    print('Estimacion: %.1f h para lo que falta\n' % (len(pendientes) * 774.3 / 3600))
_t_ini = time.time()
_tiempos = []

for k, pid in enumerate(FOLDS, 1):
    if str(pid) in hechos:
        print('[%3d/%d] %-5s  ya hecho, se omite' % (k, len(FOLDS), pid))
        continue

    t = time.time()
    semilla = SEED + k
    rng = np.random.default_rng(semilla)
    keras.backend.clear_session()
    np.random.seed(semilla); random.seed(semilla); tf.random.set_seed(semilla)

    entrenamiento = [p for p in DATOS if p != pid]
    Xtr, Ytr, rr_mu, rr_std = construir_pool(entrenamiento, rng)
    Xtr, Ytr = augmentar_multi(Xtr, Ytr, rng)

    Xte, Yte, Ste = construir_test(pid, rr_mu, rr_std)
    if len(Xte) == 0:
        print('[%3d/%d] %-5s  sin ventanas evaluables, se omite' % (k, len(FOLDS), pid))
        del Xtr, Ytr; gc.collect()
        continue

    modelo = build_cnn_gru_attn_multi((N_LOOKBACK, FEAT_DIM))
    modelo.compile(optimizer=keras.optimizers.Adam(LR_BASE), loss=ecg_loss)
    modelo.fit(Xtr, Ytr, epochs=EPOCHS_BASE, batch_size=BATCH_BASE, verbose=0)

    Yp = modelo.predict(Xte, batch_size=256, verbose=0)

    # ── error por latido, con su clase AAMI ────────────────────────────
    real = Yte.reshape(-1, BEAT_LEN)
    pred = Yp.reshape(-1, BEAT_LEN)
    dif = real - pred
    mse = np.mean(dif ** 2, axis=1)
    mae = np.mean(np.abs(dif), axis=1)
    rc = real - real.mean(axis=1, keepdims=True)
    pc = pred - pred.mean(axis=1, keepdims=True)
    den = np.sqrt((rc ** 2).sum(axis=1) * (pc ** 2).sum(axis=1))
    corr = np.where(den > 1e-8, (rc * pc).sum(axis=1) / np.maximum(den, 1e-8), 0.0)

    n_v = len(Xte)
    df = pd.DataFrame({
        'paciente': str(pid),
        'base': 'INCART' if str(pid).startswith('I') else 'MIT-BIH',
        'ventana': np.repeat(np.arange(n_v), HORIZON),
        'horizonte': np.tile(np.arange(1, HORIZON + 1), n_v),
        'clase_aami': Ste.reshape(-1),
        'mse': mse, 'rmse': np.sqrt(mse), 'mae': mae,
        'shape_corr': corr, 'shape_dist': 1.0 - corr,
    })
    r2 = r2_score(Yte.reshape(-1), Yp.reshape(-1))
    ect = int(np.isin(Ste.reshape(-1), ECTOPICAS).sum())

    # ── ORDEN DE COMPROMISO ───────────────────────────────────────────
    # 1) las ondas, 2) el CSV. Un pliegue cuenta como hecho solo si estan
    #    los dos. Si la sesion muere en medio, se repite entero.
    guardar_npz_atomico(ruta_ondas(pid),
                        y_real=Yte.astype(np.float16),
                        y_pred=Yp.astype(np.float16),
                        clase_aami=Ste,
                        paciente=np.array([str(pid)]),
                        rr_mu=np.array([rr_mu]), rr_std=np.array([rr_std]))
    guardar_csv_atomico(df, ruta_errores(pid))

    mins = (time.time() - t) / 60
    _tiempos.append(mins)
    registrar_progreso(pid, mins, r2, len(df))

    restantes = len(pendientes) - len(_tiempos)
    eta = restantes * float(np.mean(_tiempos)) / 60 if restantes > 0 else 0.0
    print('[%3d/%d] %-5s  ventanas=%5d  latidos=%6d  ectopicos=%5d  R2=%+.4f  (%.1f min)  '
          'quedan %d, faltan ~%.1f h'
          % (k, len(FOLDS), pid, n_v, len(df), ect, r2, mins, restantes, eta))

    del modelo, Xtr, Ytr, Xte, Yte, Yp, df
    gc.collect()

print('\nBucle finalizado en %.1f h.' % ((time.time() - _t_ini) / 3600))
print('  errores  ->', DIR_ERR)
print('  ondas    ->', RUTAS['ONDAS'])
print('  progreso ->', PROGRESO)
_faltan = [p for p in FOLDS if str(p) not in pliegues_completos()]
if _faltan:
    print('\nAun faltan %d pliegues: %s' % (len(_faltan), _faltan))
    print('Vuelve a ejecutar ESTA MISMA CELDA para continuar donde se quedo.')
else:
    print('\nTodos los pliegues completos.')

Pliegues: 50 totales, 11 hechos, 39 pendientes
Estimacion: 8.4 h para lo que falta

[  1/50] 102    ya hecho, se omite
[  2/50] 104    ya hecho, se omite
[  3/50] 107    ya hecho, se omite
[  4/50] 109    ya hecho, se omite
[  5/50] 114    ya hecho, se omite
[  6/50] 115    ya hecho, se omite
[  7/50] 117    ya hecho, se omite
[  8/50] 121    ya hecho, se omite
[  9/50] 122    ya hecho, se omite
[ 10/50] 123    ya hecho, se omite
[ 11/50] 124    ya hecho, se omite
[ 12/50] 203    ventanas= 2967  latidos=  8901  ectopicos= 1341  R2=+0.2960  (12.2 min)  quedan 38, faltan ~7.7 h
[ 13/50] 205    ventanas= 2647  latidos=  7941  ectopicos=  255  R2=+0.9000  (12.0 min)  quedan 37, faltan ~7.4 h
[ 14/50] 208    ventanas= 2944  latidos=  8832  ectopicos= 4085  R2=+0.5093  (11.5 min)  quedan 36, faltan ~7.1 h
[ 15/50] 212    ventanas= 2739  latidos=  8217  ectopicos=    0  R2=+0.9352  (11.7 min)  quedan 35, faltan ~6.9 h
[ 16/50] 217    ventanas= 2199  latidos=  6597  ectopicos=  486  R2=+0.5651

[ 18/50] 221    ventanas= 2418  latidos=  7254  ectopicos= 1182  R2=+0.7976  (11.1 min)  quedan 32, faltan ~6.1 h
[ 19/50] 228    ventanas= 2044  latidos=  6132  ectopicos= 1093  R2=+0.5823  (11.4 min)  quedan 31, faltan ~5.9 h


[ 20/50] 234    ventanas= 2744  latidos=  8232  ectopicos=  159  R2=+0.9766  (11.0 min)  quedan 30, faltan ~5.7 h
[ 21/50] I03    ventanas= 2443  latidos=  7329  ectopicos=  381  R2=+0.7978  (11.2 min)  quedan 29, faltan ~5.5 h
[ 22/50] I07    ventanas= 2697  latidos=  8091  ectopicos=  198  R2=+0.8629  (10.9 min)  quedan 28, faltan ~5.3 h
[ 23/50] I08    ventanas= 2121  latidos=  6363  ectopicos= 1055  R2=+0.1433  (10.8 min)  quedan 27, faltan ~5.1 h
[ 24/50] I11    ventanas= 2073  latidos=  6219  ectopicos=   12  R2=+0.9462  (11.6 min)  quedan 26, faltan ~4.9 h
[ 25/50] I12    ventanas= 2800  latidos=  8400  ectopicos=   27  R2=-0.1170  (11.7 min)  quedan 25, faltan ~4.7 h
[ 26/50] I20    ventanas= 2643  latidos=  7929  ectopicos=  867  R2=+0.3001  (12.2 min)  quedan 24, faltan ~4.6 h
[ 27/50] I21    ventanas= 2175  latidos=  6525  ectopicos=  333  R2=+0.5900  (12.7 min)  quedan 23, faltan ~4.4 h
[ 28/50] I25    ventanas= 1703  latidos=  5109  ectopicos=   19  R2=+0.6587  (11.9 min) 

### 8.1 Estado del progreso

Celda independiente: se puede ejecutar en cualquier momento, incluso en una sesión nueva tras una
caída, para ver qué hay hecho sin lanzar nada.

In [ ]:
_hechos = pliegues_completos()
_faltan = [p for p in FOLDS if str(p) not in _hechos]

print('Completos : %d de %d' % (len(FOLDS) - len(_faltan), len(FOLDS)))
print('Pendientes: %d  %s' % (len(_faltan), _faltan if _faltan else ''))

if os.path.exists(PROGRESO):
    H = json.load(open(PROGRESO, encoding='utf-8'))
    if H:
        P = pd.DataFrame(H).T
        P.index.name = 'paciente'
        print('\n%s' % P.to_string())
        _min = P['minutos'].astype(float)
        print('\nTiempo por pliegue: %.1f min de media (min %.1f, max %.1f)'
              % (_min.mean(), _min.min(), _min.max()))
        print('Invertido hasta ahora: %.1f h' % (_min.sum() / 60))
        if _faltan:
            print('Estimacion para terminar: %.1f h' % (len(_faltan) * _min.mean() / 60))
        print('R2 medio de los pliegues hechos: %.4f' % P['r2'].astype(float).mean())
else:
    print('\nAun no hay registro de progreso.')

# Comprobacion de integridad: archivos huerfanos de una interrupcion
_solo_csv, _solo_npz = [], []
for f in os.listdir(DIR_ERR):
    if f.startswith('errores_') and f.endswith('.csv'):
        p = f[len('errores_'):-len('.csv')]
        if not os.path.exists(ruta_ondas(p)):
            _solo_csv.append(p)
for f in os.listdir(RUTAS['ONDAS']):
    if f.startswith('ondas_') and f.endswith('.npz'):
        p = f[len('ondas_'):-len('.npz')]
        if not os.path.exists(ruta_errores(p)):
            _solo_npz.append(p)
if _solo_csv or _solo_npz:
    print('\nPliegues incompletos (se repetiran enteros):')
    if _solo_csv: print('  con CSV pero sin ondas :', _solo_csv)
    if _solo_npz: print('  con ondas pero sin CSV :', _solo_npz)
else:
    print('\nIntegridad correcta: ningun pliegue a medias.')

## 9. Bloque P3 — Error por clase AAMI

La pregunta: ¿el error de predicción es mayor en los latidos ectópicos que en los normales?

Ambos desenlaces son publicables. Si el error es significativamente mayor en los ectópicos, queda
acreditado que el modelo aprende el ritmo habitual del paciente y se aparta ante lo anómalo, lo que
**refuerza** el detector del bloque P4. Si es homogéneo entre clases, queda acreditada la robustez
del modelo frente a la morfología, pero **debilita** la hipótesis de detección por umbral.

In [ ]:
def cargar_errores():
    """Reune los CSV por paciente. Tolera una ejecucion parcial."""
    trozos = []
    for f in sorted(os.listdir(DIR_ERR)):
        if f.startswith('errores_') and f.endswith('.csv'):
            trozos.append(pd.read_csv(os.path.join(DIR_ERR, f), dtype={'paciente': str}))
    if not trozos:
        raise FileNotFoundError(
            'No hay ningun resultado en %s. Ejecuta antes el bucle LOPO.' % DIR_ERR)
    df = pd.concat(trozos, ignore_index=True)
    antes = len(df)
    df = df.drop_duplicates(subset=['paciente', 'ventana', 'horizonte'])
    if len(df) != antes:
        print('Se descartaron %d filas duplicadas.' % (antes - len(df)))
    return df


ERR = cargar_errores()
ERR.to_csv(os.path.join(RUTAS['SALIDA'], 'errores_por_latido.csv'), index=False)

print('Registros: %s   pacientes: %d' % (format(len(ERR), ','), ERR.paciente.nunique()))
_esperados = len([p for p in FOLDS])
if ERR.paciente.nunique() < _esperados:
    print('AVISO: hay %d pacientes de los %d previstos. El analisis es PARCIAL.'
          % (ERR.paciente.nunique(), _esperados))
    print('       Sirve para inspeccionar la tendencia, no como resultado definitivo.')

tabla = (ERR.groupby('clase_aami')
            .agg(n=('mse', 'size'), mse_media=('mse', 'mean'), mse_mediana=('mse', 'median'),
                 rmse_media=('rmse', 'mean'), mae_media=('mae', 'mean'),
                 shape_corr_media=('shape_corr', 'mean'))
            .reindex([c for c in CLASES if c in ERR.clase_aami.unique()]))
tabla['pct'] = 100.0 * tabla.n / tabla.n.sum()
print('\nERROR POR CLASE AAMI')
print(tabla.round(4).to_string())
tabla.to_csv(os.path.join(RUTAS['SALIDA'], 'p3_error_por_clase.csv'))

# Kruskal-Wallis entre las clases con muestra suficiente
grupos = [g['mse'].values for _, g in ERR.groupby('clase_aami') if len(g) >= 20]
if len(grupos) >= 3:
    H, p = kruskal(*grupos)
    print('\nKruskal-Wallis entre clases: H=%.2f  p=%.3e  %s'
          % (H, p, 'diferencias significativas' if p < 0.05 else 'sin diferencias'))

# Contraste normal frente a ectopico
normales = ERR.loc[ERR.clase_aami == 'N', 'mse'].values
ectopicos = ERR.loc[ERR.clase_aami.isin(ECTOPICAS), 'mse'].values
if len(normales) > 10 and len(ectopicos) > 10:
    U, p = mannwhitneyu(normales, ectopicos, alternative='two-sided')
    razon = ectopicos.mean() / normales.mean() if normales.mean() > 0 else float('nan')
    print('\nNORMAL frente a ECTOPICO')
    print('  normales : n=%s  MSE medio=%.4f' % (format(len(normales), ','), normales.mean()))
    print('  ectopicos: n=%s  MSE medio=%.4f' % (format(len(ectopicos), ','), ectopicos.mean()))
    print('  razon ectopico/normal: %.2fx' % razon)
    print('  Mann-Whitney U=%.1f  p=%.3e  %s'
          % (U, p, 'SIGNIFICATIVO' if p < 0.05 else 'no significativo'))
    json.dump({'mse_normal': float(normales.mean()), 'mse_ectopico': float(ectopicos.mean()),
               'razon': float(razon), 'U': float(U), 'p': float(p),
               'n_normal': int(len(normales)), 'n_ectopico': int(len(ectopicos))},
              open(os.path.join(RUTAS['SALIDA'], 'p3_contraste.json'), 'w'), indent=1)

## 10. Bloque P4 — Detección de evento a partir del error

Etiqueta binaria: **normal** (clase N) frente a **ectópico** (SVEB, VEB o F). La clase Q se excluye
por corresponder a latidos no clasificables y estimulados, que no constituyen un evento arrítmico
espontáneo.

El umbral se calcula **sin fuga**: para cada paciente se obtiene sobre los demás y se aplica a él.
En ningún momento se optimiza sobre el paciente evaluado.

In [ ]:
E = ERR[ERR.clase_aami != 'Q'].copy()
E['y'] = E.clase_aami.isin(ECTOPICAS).astype(int)

prevalencia = E.y.mean()
print('Latidos evaluables: %s' % format(len(E), ','))
print('Ectopicos: %s  (%.2f %% de prevalencia)' % (format(int(E.y.sum()), ','), 100 * prevalencia))
print('Pacientes: %d' % E.paciente.nunique())

if E.y.sum() < 20:
    print('\nAVISO: hay muy pocos latidos ectopicos para un contraste fiable.')
    print('       Con MODO_PRUEBA = True esto es lo esperable.')

In [ ]:
def umbral_optimo(y, sc):
    """Umbral que maximiza F1 sobre (y, sc). Devuelve None si no es calculable."""
    if len(np.unique(y)) < 2:
        return None
    prec, rec, umb = precision_recall_curve(y, sc)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros_like(prec), where=(prec + rec) > 0)
    if len(umb) == 0:
        return None
    return float(umb[min(int(np.argmax(f1)), len(umb) - 1)])


PUNTUACION = 'mse'   # el residuo de prediccion es el indicador
resultados = []
for pid, sub in E.groupby('paciente'):
    resto = E[E.paciente != pid]
    u = umbral_optimo(resto.y.values, resto[PUNTUACION].values)
    if u is None or len(np.unique(sub.y.values)) < 2:
        continue
    pred = (sub[PUNTUACION].values >= u).astype(int)
    vp = int(((pred == 1) & (sub.y.values == 1)).sum())
    fp = int(((pred == 1) & (sub.y.values == 0)).sum())
    fn = int(((pred == 0) & (sub.y.values == 1)).sum())
    vn = int(((pred == 0) & (sub.y.values == 0)).sum())
    precision = vp / (vp + fp) if (vp + fp) else 0.0
    exhaustividad = vp / (vp + fn) if (vp + fn) else 0.0
    f1 = (2 * precision * exhaustividad / (precision + exhaustividad)
          if (precision + exhaustividad) else 0.0)
    especificidad = vn / (vn + fp) if (vn + fp) else 0.0
    resultados.append(dict(paciente=pid, base=sub.base.iloc[0], umbral=u,
                           n=len(sub), n_ectopicos=int(sub.y.sum()),
                           VP=vp, FP=fp, FN=fn, VN=vn,
                           precision=precision, exhaustividad=exhaustividad,
                           especificidad=especificidad, f1=f1,
                           ap=float(average_precision_score(sub.y.values,
                                                            sub[PUNTUACION].values))))

DET = pd.DataFrame(resultados)
if len(DET):
    DET.to_csv(os.path.join(RUTAS['SALIDA'], 'p4_deteccion_por_paciente.csv'), index=False)
    print('\nDETECCION POR PACIENTE (umbral anidado, sin fuga)')
    print(DET[['paciente', 'base', 'n', 'n_ectopicos', 'precision',
               'exhaustividad', 'especificidad', 'f1', 'ap']].round(4).to_string(index=False))
    print('\nMEDIA ENTRE PACIENTES')
    for c in ['precision', 'exhaustividad', 'especificidad', 'f1', 'ap']:
        print('  %-14s %.4f  (desv %.4f)' % (c, DET[c].mean(), DET[c].std(ddof=1)))
else:
    print('\nNo se pudo evaluar ningun paciente: hacen falta ambas clases en cada uno.')

### 10.1 Curva de precisión-exhaustividad

El criterio de éxito es que el **AUC-PR supere de manera holgada la prevalencia** de la clase
ectópica. Un AUC-PR próximo a la prevalencia indicaría ausencia de capacidad discriminante.

Se usa precisión-exhaustividad y no la curva ROC porque, con la clase positiva en torno al 14 %,
la ROC produce valores optimistas que no reflejan el desempeño real.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

y = E.y.values
sc = E[PUNTUACION].values
auc_pr = average_precision_score(y, sc)
prec, rec, _ = precision_recall_curve(y, sc)

print('AUC-PR global      : %.4f' % auc_pr)
print('Prevalencia (base) : %.4f' % prevalencia)
print('Ganancia           : %.2fx sobre la linea base' % (auc_pr / prevalencia))
print('Veredicto          : %s'
      % ('capacidad discriminante' if auc_pr > 1.5 * prevalencia else 'sin capacidad util'))

fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=140)
ax.plot(rec, prec, lw=2, color='#0B5E77', label='Residuo de prediccion (AUC-PR = %.3f)' % auc_pr)
ax.axhline(prevalencia, ls='--', lw=1.2, color='#A81F2D',
           label='Prevalencia = %.3f' % prevalencia)
ax.set_xlabel('Exhaustividad'); ax.set_ylabel('Precision')
ax.set_title('Deteccion de latido ectopico a partir del residuo')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=8); ax.grid(alpha=.25)
fig.tight_layout()
_png = os.path.join(RUTAS['SALIDA'], 'p4_curva_precision_exhaustividad.png')
fig.savefig(_png); plt.close(fig)
print('\nFigura guardada en', _png)

json.dump({'auc_pr': float(auc_pr), 'prevalencia': float(prevalencia),
           'ganancia': float(auc_pr / prevalencia), 'puntuacion': PUNTUACION,
           'n_latidos': int(len(E)), 'n_ectopicos': int(E.y.sum()),
           'n_pacientes': int(E.paciente.nunique())},
          open(os.path.join(RUTAS['SALIDA'], 'p4_resumen.json'), 'w'), indent=1)
print('Resumen guardado.')

## 11. Salidas y dónde va cada resultado

| Archivo | Contenido | Destino en el documento |
| :---- | :---- | :---- |
| `errores_por_paciente/errores_*.csv` | un archivo por pliegue: error, fidelidad de forma y clase AAMI de cada latido | insumo, resistente a caídas |
| `errores_por_latido.csv` | los anteriores reunidos y sin duplicados | tabla de base de los dos bloques |
| `progreso.json` | tiempo, R² y latidos de cada pliegue completado | control de avance y reanudación |
| `salvedad_derivaciones.json` | registros cuyo canal 0 no es MLII | limitación a declarar en el documento |
| `p3_error_por_clase.csv` | error agregado por clase AAMI | sustituye el bloque `[PENDIENTE — P3]` |
| `p3_contraste.json` | contraste normal frente a ectópico | mismo bloque |
| `p4_deteccion_por_paciente.csv` | precisión, exhaustividad, especificidad y F1 por paciente | sustituye el bloque `[PENDIENTE — P4]` |
| `p4_curva_precision_exhaustividad.png` | curva PR con la línea de prevalencia | figura del bloque P4 |
| `p4_resumen.json` | AUC-PR, prevalencia y ganancia | mismo bloque |
| `ondas_predichas/ondas_*.npz` | formas de onda real y predicha, con su clase | **insumo para el clasificador externo** |

## Sobre el clasificador de arritmias externo

Cada archivo `ondas_*.npz` contiene `y_real` y `y_pred` con forma `(ventanas, 3, 256)` y
`clase_aami` con forma `(ventanas, 3)`. Ambas señales están normalizadas por latido con z-score y
recorte a ±5, en la misma escala.

El protocolo correcto tiene tres pasos, y el segundo es imprescindible:

1. **Control.** Pasar el clasificador por `y_real`. Establece su exactitud en este dominio.
   Si aquí ya cae mucho, el desajuste de dominio invalida todo lo que venga después.
2. **Experimento.** Pasar el clasificador por `y_pred`.
3. **Comparación.** Matriz de confusión entre ambas salidas, y concordancia por clase.

Dos cautelas. La primera: comprobar con qué pacientes se entrenó ese clasificador; si vio los
mismos que aquí se evalúan, hay fuga de información. La segunda, de fondo: un latido ectópico es
prematuro y anómalo por definición, de modo que el modelo, alimentado con cinco latidos normales,
tenderá a predecir un sexto normal. Que el clasificador diga «normal» sobre esa predicción **no es
un fallo del experimento**: es justamente lo que hace que el residuo se dispare, que es el
mecanismo de detección del bloque P4. Los dos análisis se refuerzan.